In [1]:
# -------------------------------------------------
# 0. SET PROJECT ROOT (run once)
# -------------------------------------------------
import os, sys

# Go up one level from notebooks/ → flood-mumbai/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)
sys.path.append(PROJECT_ROOT)

print("Working directory set to:", os.getcwd())

Working directory set to: /Users/vainavilad/flood-mumbai


In [3]:
import os, sys, numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score

In [4]:
# -------------------------------------------------
# 1. LOAD images.npy and masks.npy
# -------------------------------------------------
DATA_DIR = "data/processed"
images = np.load(os.path.join(DATA_DIR, "images.npy"))   # (N, 256, 256, 2)
masks  = np.load(os.path.join(DATA_DIR, "masks.npy"))    # (N, 256, 256, 1) or (N, 256, 256)

print(f"Loaded {len(images)} images, {len(masks)} masks")

# Ensure masks have channel dim
if masks.ndim == 3:
    masks = np.expand_dims(masks, -1)
masks = masks.astype(np.float32)

Loaded 900 images, 900 masks


In [5]:
# -------------------------------------------------
# 2. RECREATE STRATIFIED TRAIN/VAL SPLIT
# -------------------------------------------------
flood_ratios = np.mean(masks, axis=(1,2,3))  # (N,)
labels = np.digitize(flood_ratios, bins=[0.01, 0.1, 0.3]).astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    images, masks,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

# Save val split for dashboard
np.save(os.path.join(DATA_DIR, "X_val.npy"), X_val)
np.save(os.path.join(DATA_DIR, "y_val.npy"), y_val)
print(f"Split: {len(X_train)} train, {len(X_val)} val")

Split: 720 train, 180 val


In [6]:
# -------------------------------------------------
# 3. CUSTOM METRICS
# -------------------------------------------------
def dice_coefficient(y_true, y_pred):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    inter = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * inter + 1.) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + 1.)

def iou_metric(y_true, y_pred):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.round(tf.reshape(tf.cast(y_pred, tf.float32), [-1]))
    inter = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - inter
    return inter / (union + 1e-7)

In [7]:
# -------------------------------------------------
# 4. LOAD MODELS
# -------------------------------------------------
MODEL_DIR = "models"

def safe_load(name):
    path = os.path.join(MODEL_DIR, name)
    if not os.path.isdir(path):
        print(f"NOT FOUND: {path}")
        return None
    try:
        return load_model(path, custom_objects={'dice_coefficient': dice_coefficient, 'iou_metric': iou_metric}, compile=False)
    except Exception as e:
        print(f"ERROR loading {name}: {e}")
        return None

model_unet    = safe_load("unet_flood_savedmodel")
model_deeplab = safe_load("deeplab_macair_flood_savedmodel")

if not (model_unet and model_deeplab):
    raise RuntimeError("Both models must be in models/ folder")

print("Both models loaded!")

The dtype policy mixed_float16 may run slowly because this machine does not have a GPU. Only Nvidia GPUs with compute capability of at least 7.0 run quickly with mixed_float16.
If you will use compatible GPU(s) not attached to this host, e.g. by running a multi-worker model, you can ignore this warning. This message will only be logged once
Both models loaded!


In [8]:
# -------------------------------------------------
# 5. PREDICT ON VAL SET
# -------------------------------------------------
print(f"Predicting on {len(X_val)} validation tiles...")
pred_unet    = (model_unet.predict(X_val, batch_size=4, verbose=0) > 0.5).astype(np.uint8)
pred_deeplab = (model_deeplab.predict(X_val, batch_size=4, verbose=0) > 0.5).astype(np.uint8)

Predicting on 180 validation tiles...


In [10]:
# -------------------------------------------------
# 6. COMPUTE METRICS
# -------------------------------------------------
def compute_metrics(y_true, y_pred):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    dice = 2*np.sum(y_true_f*y_pred_f)/(np.sum(y_true_f)+np.sum(y_pred_f)+1e-7)
    iou  = np.sum(y_true_f*y_pred_f)/(np.sum(y_true_f)+np.sum(y_pred_f)-np.sum(y_true_f*y_pred_f)+1e-7)
    acc  = accuracy_score(y_true_f, y_pred_f)
    return {'dice': dice, 'iou': iou, 'accuracy': acc}

metrics = {
    'unet':    compute_metrics(y_val, pred_unet),
    'deeplab': compute_metrics(y_val, pred_deeplab)
}

In [11]:
# -------------------------------------------------
# 7. SAVE FOR DASHBOARD
# -------------------------------------------------
PROC_DIR = "data/processed"
os.makedirs(PROC_DIR, exist_ok=True)

np.save(os.path.join(PROC_DIR, "pred_unet.npy"),    pred_unet)
np.save(os.path.join(PROC_DIR, "pred_deeplab.npy"), pred_deeplab)
np.save(os.path.join(PROC_DIR, "metrics.npy"),      metrics)
np.save(os.path.join(PROC_DIR, "sample_indices.npy"), np.arange(len(X_val)))

print("\nSAVED FOR DASHBOARD:")
print("  pred_unet.npy    :", pred_unet.shape)
print("  pred_deeplab.npy :", pred_deeplab.shape)
print("  metrics.npy")
print("  X_val.npy        :", X_val.shape)
print("  y_val.npy        :", y_val.shape)

print("\nU-Net Dice   :", f"{metrics['unet']['dice']:.4f}")
print("DeepLab Dice :", f"{metrics['deeplab']['dice']:.4f}")


SAVED FOR DASHBOARD:
  pred_unet.npy    : (180, 256, 256, 1)
  pred_deeplab.npy : (180, 256, 256, 1)
  metrics.npy
  X_val.npy        : (180, 256, 256, 2)
  y_val.npy        : (180, 256, 256, 1)

U-Net Dice   : 0.8071
DeepLab Dice : 0.8256
